# YOLOv12 Bangladesh Street Food Detection Training

[![Roboflow](https://img.shields.io/badge/Roboflow-Dataset-blue)](https://roboflow.com)
[![YOLOv12](https://img.shields.io/badge/YOLOv12-Model-green)](https://github.com/sunsmarterjie/yolov12)
[![arXiv](https://img.shields.io/badge/arXiv-2502.12524-b31b1b.svg)](https://arxiv.org/abs/2502.12524)

This notebook demonstrates how to train a YOLOv12 model for detecting Bangladesh street food items. We'll focus on 3 popular street foods: **Fuska**, **Singara**, and **Jhalmuri**.

## Key Features of YOLOv12
- **Attention-centric architecture** with area attention modules
- **Residual Efficient Layer Aggregation Networks (R-ELAN)**
- **Better accuracy-speed trade-off** compared to YOLOv11/YOLOv8
- **FlashAttention optimization** for modern GPUs

![YOLOv12 Architecture](https://media.roboflow.com/notebooks/examples/yolov12-area-attention.png)

## Environment Setup

### Check GPU Availability

**Important:** YOLOv12 leverages FlashAttention for optimal performance, requiring NVIDIA GPUs with Ampere architecture or newer (RTX 30xx/40xx, A100, etc.).

In [ ]:
!nvidia-smi

In [ ]:
import os
import torch

HOME = os.getcwd()
print(f"Working directory: {HOME}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)
    ampere_compatible = gpu_capability[0] >= 8
    
    print(f"GPU: {gpu_name}")
    print(f"Compute Capability: {gpu_capability}")
    print(f"FlashAttention Compatible: {'✅' if ampere_compatible else '❌'}")
    
    if not ampere_compatible:
        print("⚠️ FlashAttention will be disabled for this GPU")
else:
    print("⚠️ No GPU detected. Training will be slow on CPU.")

### Configure API Keys

Set up your Roboflow API key to access the Bangladesh Street Food dataset:
1. Go to [Roboflow Settings](https://app.roboflow.com/settings/api)
2. Copy your API key
3. Set it as an environment variable or enter it below

In [ ]:
import os
from getpass import getpass

# Try to get API key from environment, otherwise prompt for it
ROBOFLOW_API_KEY = os.getenv('ROBOFLOW_API_KEY')

if not ROBOFLOW_API_KEY:
    print("Roboflow API key not found in environment variables.")
    ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")

os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
print(f"✅ API key configured: {ROBOFLOW_API_KEY[:8]}...")

### Install YOLOv12 and Dependencies

YOLOv12 is installed directly from GitHub along with required dependencies.

In [ ]:
# Note: YOLOv12 installation command corrected based on official repository
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git
!pip install -q roboflow supervision wandb python-dotenv

# Install FlashAttention (may fail on incompatible GPUs)
# FlashAttention is not required for YOLOv12 basic functionality
try:
    !pip install -q flash-attn>=2.0.0 --no-build-isolation
    print("✅ FlashAttention installed successfully")
except Exception as e:
    print(f"⚠️ FlashAttention installation failed: {e}")
    print("⚠️ Continuing without FlashAttention optimization")

### Import Libraries

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import yaml
import shutil
import random
import time
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO
import supervision as sv
from roboflow import Roboflow

print("✅ All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")

## Bangladesh Street Food Dataset Configuration

We'll work with a curated dataset of 15 popular Bangladesh street foods and focus on training for 3 specific items.

In [ ]:
# Bangladesh Street Food Classes
ALL_BANGLADESH_FOODS = [
    'Tiler Khaja', 'Chanachur', 'Jhalmuri', 'Peyaju', 'Beguni', 'Singara', 
    'Papor Vaja', 'Vel Puri', 'Chotpoti', 'Fuska', 'Vorta', 'Murobba', 
    'Dim Cake', 'Halim', 'Puri'
]

# Selected 3 classes for focused training
SELECTED_CLASSES = [
    'Fuska',      # Popular street snack with distinct round shape
    'Singara',    # Triangular samosa-like item with clear geometry 
    'Jhalmuri'    # Mixed puffed rice with diverse textures
]

print(f"🍛 Bangladesh Street Food Detection")
print(f"Total available classes: {len(ALL_BANGLADESH_FOODS)}")
print(f"Selected for training: {SELECTED_CLASSES}")
print(f"Number of classes: {len(SELECTED_CLASSES)}")

# Dataset configuration
ROBOFLOW_WORKSPACE = "bangladesh-street-food"  # Update with your workspace
ROBOFLOW_PROJECT = "street-food-detection"     # Update with your project
DATASET_VERSION = 1

## Download Dataset from Roboflow

In [ ]:
def download_bangladesh_dataset():
    """Download Bangladesh Street Food dataset from Roboflow"""
    try:
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
        
        # Download in YOLOv8 format (compatible with YOLOv12)
        dataset = project.version(DATASET_VERSION).download("yolov8")
        
        print(f"✅ Dataset downloaded successfully")
        print(f"Location: {dataset.location}")
        return dataset.location
        
    except Exception as e:
        print(f"❌ Failed to download dataset: {e}")
        print("Please check your API key and project details")
        return None

# Download dataset
dataset_path = download_bangladesh_dataset()

if dataset_path:
    !ls {dataset_path}
else:
    print("Please check your Roboflow configuration and try again")

### Prepare Dataset for YOLOv12

Update the data.yaml file to ensure compatibility with YOLOv12 training.

In [ ]:
if dataset_path:
    # Fix data.yaml paths for YOLOv12 compatibility
    data_yaml_path = f"{dataset_path}/data.yaml"
    
    # Read current data.yaml
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print("Original data.yaml:")
    print(yaml.dump(data_config, default_flow_style=False))
    
    # Update for YOLOv12 compatibility
    data_config.update({
        'path': dataset_path,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images'
    })
    
    # Write updated data.yaml
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)
    
    print("\nUpdated data.yaml:")
    !cat {data_yaml_path}
else:
    print("No dataset available to process")

### Filter Dataset to 3 Classes

Create a focused dataset with only our selected 3 Bangladesh street food classes.

In [ ]:
def filter_dataset_for_3_classes(dataset_path, selected_classes):
    """Filter dataset to keep only selected 3 classes"""
    if not dataset_path or not os.path.exists(dataset_path):
        return None, None
    
    print(f"🔍 Filtering dataset for: {selected_classes}")
    
    # Create filtered dataset directory
    base_name = os.path.basename(dataset_path.rstrip('/'))
    filtered_path = os.path.join(os.path.dirname(dataset_path), f"{base_name}_3classes")
    
    if os.path.exists(filtered_path):
        shutil.rmtree(filtered_path)
    os.makedirs(filtered_path)
    
    # Read original data.yaml
    with open(f"{dataset_path}/data.yaml", 'r') as f:
        original_data = yaml.safe_load(f)
    
    original_classes = original_data['names']
    print(f"Original classes ({len(original_classes)}): {original_classes}")
    
    # Create class mapping
    original_class_to_idx = {name: idx for idx, name in enumerate(original_classes)}
    selected_indices = [original_class_to_idx[cls] for cls in selected_classes if cls in original_class_to_idx]
    new_class_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(selected_indices)}
    
    print(f"Selected indices: {selected_indices}")
    print(f"Class mapping: {new_class_mapping}")
    
    # Statistics
    stats = defaultdict(lambda: defaultdict(int))
    
    # Process each split
    for split in ['train', 'valid', 'test']:
        split_src = f"{dataset_path}/{split}"
        split_dst = f"{filtered_path}/{split}"
        
        if not os.path.exists(split_src):
            continue
        
        # Create directories
        os.makedirs(f"{split_dst}/images", exist_ok=True)
        os.makedirs(f"{split_dst}/labels", exist_ok=True)
        
        images_src = f"{split_src}/images"
        labels_src = f"{split_src}/labels"
        
        if not os.path.exists(images_src) or not os.path.exists(labels_src):
            continue
        
        image_files = [f for f in os.listdir(images_src) if f.endswith(('.jpg', '.jpeg', '.png'))]
        kept_images = 0
        
        for image_file in image_files:
            label_file = os.path.splitext(image_file)[0] + '.txt'
            image_path = f"{images_src}/{image_file}"
            label_path = f"{labels_src}/{label_file}"
            
            if not os.path.exists(label_path):
                continue
            
            # Read and filter labels
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            new_labels = []
            has_selected_class = False
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    if class_id in new_class_mapping:
                        new_class_id = new_class_mapping[class_id]
                        new_line = f"{new_class_id} {' '.join(parts[1:])}\n"
                        new_labels.append(new_line)
                        has_selected_class = True
                        stats[split][selected_classes[new_class_id]] += 1
            
            # Keep images with selected classes
            if has_selected_class:
                shutil.copy2(image_path, f"{split_dst}/images/{image_file}")
                with open(f"{split_dst}/labels/{label_file}", 'w') as f:
                    f.writelines(new_labels)
                kept_images += 1
        
        print(f"  {split}: kept {kept_images}/{len(image_files)} images")
    
    # Create new data.yaml
    new_data_yaml = {
        'path': filtered_path,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(selected_classes),
        'names': selected_classes
    }
    
    with open(f"{filtered_path}/data.yaml", 'w') as f:
        yaml.dump(new_data_yaml, f, default_flow_style=False)
    
    # Print statistics
    print(f"\n📊 FILTERED DATASET STATISTICS")
    for split in stats:
        print(f"\n{split.upper()}:")
        total = sum(stats[split].values())
        for class_name, count in stats[split].items():
            pct = (count/total*100) if total > 0 else 0
            print(f"  {class_name}: {count} objects ({pct:.1f}%)")
        print(f"  Total: {total} objects")
    
    return filtered_path, stats

# Filter dataset if it has more than 3 classes
if dataset_path:
    with open(f"{dataset_path}/data.yaml", 'r') as f:
        data_info = yaml.safe_load(f)
    
    if len(data_info['names']) > 3:
        print(f"Filtering from {len(data_info['names'])} classes to 3...")
        filtered_dataset_path, filter_stats = filter_dataset_for_3_classes(dataset_path, SELECTED_CLASSES)
        if filtered_dataset_path:
            dataset_path = filtered_dataset_path
            print(f"✅ Using filtered dataset: {dataset_path}")
    else:
        print(f"Dataset already has {len(data_info['names'])} classes, no filtering needed")

## Test YOLOv12 Inference

Before training, let's test YOLOv12 inference on a sample image to ensure everything is working.

In [ ]:
# Download a test image
!wget -q https://media.roboflow.com/notebooks/examples/dog.jpeg -O test_image.jpg

# Test YOLOv12 inference
print("🧪 Testing YOLOv12 inference...")

try:
    # Load YOLOv12 model
    test_model = YOLO('yolov12l.pt')
    
    # Load and process image
    image_path = "test_image.jpg"
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Run inference
    results = test_model(image_path, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    
    # Visualize results
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    
    annotated_image = image_rgb.copy()
    annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)
    annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)
    
    # Display
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(image_rgb)
    plt.title("Original Image")
    plt.axis('off')
    
    plt.subplot(1, 2, 2) 
    plt.imshow(annotated_image)
    plt.title("YOLOv12 Detection Results")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✅ YOLOv12 test successful! Detected {len(detections)} objects")
    
    # Clean up
    del test_model
    
except Exception as e:
    print(f"❌ YOLOv12 test failed: {e}")
    print("Please check your installation")

## YOLOv12 Model Training Configuration

Configure YOLOv12 training parameters optimized for Bangladesh street food detection.

In [ ]:
YOLOV12_CONFIG = {
    # Model Configuration
    'model_size': 'yolov12l',  # Options: yolov12n, yolov12s, yolov12m, yolov12l, yolov12x
    'model_weights': 'yolov12l.pt',  # Use .pt for transfer learning (recommended)
    
    # Training Parameters
    'epochs': 100,
    'batch_size': 16,  # Adjust based on GPU memory
    'imgsz': 640,
    'patience': 30,
    'save_period': 10,
    'workers': 8,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    
    # Project Settings
    'project': 'yolov12_bangladesh_street_food',
    'name': f'yolov12l_3classes_{"_".join(SELECTED_CLASSES)}',
    'exist_ok': True,
    
    # Optimization (Updated for YOLOv12)
    'optimizer': 'AdamW',
    'lr0': 0.01,       # Standard learning rate for YOLOv12
    'lrf': 0.001,      # Final learning rate factor
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    
    # Loss Configuration (YOLOv12 specific)
    'box': 7.5,
    'cls': 0.5,        # Standard for multi-class problems
    'dfl': 1.5,
    'label_smoothing': 0.0,  # Disabled for small dataset
    
    # Performance
    'amp': True,       # Automatic Mixed Precision
    'half': False,     # Disabled for stability
    'cache': True,     # Cache images in memory
    'cos_lr': True,    # Cosine LR scheduler
    'close_mosaic': 15,
    
    # Validation
    'val': True,
    'plots': True,
    'save_json': True,
    'conf': 0.001,     # Lower confidence for validation
    'iou': 0.6,        # Standard NMS IoU threshold
    'max_det': 300,
    
    # Augmentation (optimized for food items)
    'hsv_h': 0.015,    # Hue
    'hsv_s': 0.7,      # Saturation
    'hsv_v': 0.4,      # Value
    'degrees': 0.0,    # No rotation for food
    'translate': 0.1,  # Translation
    'scale': 0.5,      # Scale variation
    'shear': 0.0,      # No shear
    'perspective': 0.0, # No perspective
    'flipud': 0.0,     # No vertical flip
    'fliplr': 0.5,     # Horizontal flip
    'mosaic': 1.0,     # Mosaic probability
    'mixup': 0.0,      # Disabled for small dataset
    'copy_paste': 0.0, # Disabled for 3-class problem
}

print(f"🔧 YOLOv12 Training Configuration:")
print(f"Model: {YOLOV12_CONFIG['model_size']}")
print(f"Epochs: {YOLOV12_CONFIG['epochs']}")
print(f"Batch Size: {YOLOV12_CONFIG['batch_size']}")
print(f"Image Size: {YOLOV12_CONFIG['imgsz']}")
print(f"Device: {YOLOV12_CONFIG['device']}")
print(f"Classes: {len(SELECTED_CLASSES)} ({', '.join(SELECTED_CLASSES)})")
print(f"Learning Rate: {YOLOV12_CONFIG['lr0']}")
print(f"Optimizer: {YOLOV12_CONFIG['optimizer']}")

## Train YOLOv12 Model

Now we'll train the YOLOv12 model on our Bangladesh street food dataset.

In [ ]:
def train_yolov12_bangladesh_food(dataset_path, config):
    """Train YOLOv12 on Bangladesh Street Food dataset"""
    
    print(f"🚀 Starting YOLOv12 Training")
    print(f"{'='*50}")
    print(f"Dataset: {dataset_path}")
    print(f"Model: {config['model_size']}")
    print(f"Classes: {SELECTED_CLASSES}")
    
    # Initialize model - Use pretrained weights for better results
    try:
        model = YOLO(config['model_weights'])
        print(f"✅ YOLOv12 model initialized with pretrained weights: {config['model_weights']}")
    except Exception as e:
        print(f"⚠️ Failed to load pretrained weights, trying YAML: {e}")
        try:
            model = YOLO(f"{config['model_size']}.yaml")
            print(f"✅ YOLOv12 model initialized from YAML")
        except Exception as e2:
            print(f"❌ Failed to initialize model: {e2}")
            return None, None
    
    # Training arguments (corrected parameter names)
    train_args = {
        'data': f"{dataset_path}/data.yaml",
        'epochs': config['epochs'],
        'batch': config['batch_size'],
        'imgsz': config['imgsz'],
        'patience': config['patience'],
        'save_period': config['save_period'],
        'workers': config['workers'],
        'device': config['device'],
        'project': config['project'],
        'name': config['name'],
        'exist_ok': config['exist_ok'],
        'optimizer': config['optimizer'],
        'lr0': config['lr0'],
        'lrf': config['lrf'],
        'momentum': config['momentum'],
        'weight_decay': config['weight_decay'],
        'warmup_epochs': config['warmup_epochs'],
        'box': config['box'],
        'cls': config['cls'],
        'dfl': config['dfl'],
        'amp': config['amp'],
        'cache': config['cache'],
        'cos_lr': config['cos_lr'],
        'val': config['val'],
        'plots': config['plots'],
        'conf': config['conf'],
        'iou': config['iou'],
        'hsv_h': config['hsv_h'],
        'hsv_s': config['hsv_s'],
        'hsv_v': config['hsv_v'],
        'degrees': config['degrees'],
        'translate': config['translate'],
        'scale': config['scale'],
        'fliplr': config['fliplr'],
        'mosaic': config['mosaic'],
    }
    
    # Add optional parameters only if they're not disabled
    if config.get('mixup', 0) > 0:
        train_args['mixup'] = config['mixup']
    if config.get('copy_paste', 0) > 0:
        train_args['copy_paste'] = config['copy_paste']
    if config.get('label_smoothing', 0) > 0:
        train_args['label_smoothing'] = config['label_smoothing']
    
    # Start training with timing
    start_time = time.time()
    print(f"\n⏱️ Training started at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    try:
        results = model.train(**train_args)
        training_time = time.time() - start_time
        
        print(f"\n✅ Training completed successfully!")
        print(f"Training time: {training_time/3600:.2f} hours")
        print(f"Results saved to: {config['project']}/{config['name']}")
        
        return model, results
        
    except Exception as e:
        print(f"❌ Training failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# GPU memory optimization
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print("✅ GPU memory optimized")

# Start training
if dataset_path and os.path.exists(dataset_path):
    trained_model, training_results = train_yolov12_bangladesh_food(dataset_path, YOLOV12_CONFIG)
    
    if trained_model:
        best_model_path = f"{YOLOV12_CONFIG['project']}/{YOLOV12_CONFIG['name']}/weights/best.pt"
        print(f"\n💾 Best model saved: {best_model_path}")
    else:
        print("❌ Training failed")
else:
    print("❌ No dataset available for training")

## Evaluate YOLOv12 Model

Analyze the training results and evaluate model performance.

In [ ]:
# Display training results
if 'best_model_path' in locals() and os.path.exists(best_model_path):
    results_dir = f"{YOLOV12_CONFIG['project']}/{YOLOV12_CONFIG['name']}"
    
    print(f"📊 Training Results Directory: {results_dir}")
    !ls {results_dir}
    
    # Display training curves
    results_plot = f"{results_dir}/results.png"
    if os.path.exists(results_plot):
        from IPython.display import Image, display
        display(Image(filename=results_plot, width=1000))
    
    # Display confusion matrix
    confusion_matrix = f"{results_dir}/confusion_matrix.png"
    if os.path.exists(confusion_matrix):
        display(Image(filename=confusion_matrix, width=800))
else:
    print("No trained model available for evaluation")

### Comprehensive Model Evaluation with Supervision

In [ ]:
if 'best_model_path' in locals() and os.path.exists(best_model_path):
    print("🔍 Comprehensive Model Evaluation")
    
    # Load the trained model
    model = YOLO(best_model_path)
    
    # Load test dataset using supervision
    try:
        ds = sv.DetectionDataset.from_yolo(
            images_directory_path=f"{dataset_path}/test/images",
            annotations_directory_path=f"{dataset_path}/test/labels",
            data_yaml_path=f"{dataset_path}/data.yaml"
        )
        
        print(f"Test dataset loaded: {len(ds)} images")
        print(f"Classes: {ds.classes}")
        
        # Run inference on test set
        predictions = []
        targets = []
        
        print("Running inference on test set...")
        for i, (_, image, target) in enumerate(ds):
            if i % 20 == 0:
                print(f"  Processed {i}/{len(ds)} images")
            
            results = model(image, verbose=False)[0]
            detections = sv.Detections.from_ultralytics(results)
            
            predictions.append(detections)
            targets.append(target)
        
        # Calculate metrics
        from supervision.metrics import MeanAveragePrecision
        map_metric = MeanAveragePrecision().update(predictions, targets).compute()
        
        print(f"\n📈 YOLOv12 Bangladesh Street Food Detection Results:")
        print(f"mAP 50-95: {map_metric.map50_95:.4f}")
        print(f"mAP 50: {map_metric.map50:.4f}")
        print(f"mAP 75: {map_metric.map75:.4f}")
        
        # Plot results
        try:
            fig = map_metric.plot()
            plt.title("YOLOv12 Bangladesh Street Food Detection - mAP Results")
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Could not display mAP plot: {e}")
            
    except Exception as e:
        print(f"❌ Evaluation failed: {e}")
else:
    print("No trained model available for evaluation")

## Run Inference with Trained YOLOv12 Model

Test the trained model on sample images from our Bangladesh street food dataset.

In [ ]:
if 'best_model_path' in locals() and os.path.exists(best_model_path):
    print("🎯 YOLOv12 Inference Demo on Bangladesh Street Food")
    
    # Load trained model
    model = YOLO(best_model_path)
    
    # Load test dataset for visualization
    ds = sv.DetectionDataset.from_yolo(
        images_directory_path=f"{dataset_path}/test/images",
        annotations_directory_path=f"{dataset_path}/test/labels",
        data_yaml_path=f"{dataset_path}/data.yaml"
    )
    
    # Select random test images
    num_samples = min(3, len(ds))
    sample_indices = random.sample(range(len(ds)), num_samples)
    
    print(f"Running inference on {num_samples} sample images...")
    
    for i, idx in enumerate(sample_indices):
        image_path, image, target = ds[idx]
        
        # Run inference
        results = model(img_path, conf=0.25, iou=0.45, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        
        # Apply NMS if needed
        if len(detections) > 0:
            detections = detections.with_nms(threshold=0.45)
        
        # Annotate image
        box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.CLASS)
        label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=0.8, color_lookup=sv.ColorLookup.CLASS)
        
        annotated_image = image.copy()
        annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)
        
        # Create labels with class names and confidence
        labels = []
        for class_id, confidence in zip(detections.class_id, detections.confidence):
            class_name = SELECTED_CLASSES[class_id] if class_id < len(SELECTED_CLASSES) else f"Class_{class_id}"
            labels.append(f"{class_name} {confidence:.2f}")
        
        annotated_image = label_annotator.annotate(
            scene=annotated_image, 
            detections=detections, 
            labels=labels
        )
        
        # Display results
        plt.figure(figsize=(15, 6))
        
        # Original image
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title(f"Original Image {i+1}")
        plt.axis('off')
        
        # Annotated image
        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
        plt.title(f"YOLOv12 Bangladesh Street Food Detection")
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print detection details
        if len(detections) > 0:
            print(f"\nDetections in Image {i+1}:")
            for j, (class_id, conf) in enumerate(zip(detections.class_id, detections.confidence)):
                class_name = SELECTED_CLASSES[class_id] if class_id < len(SELECTED_CLASSES) else f"Class_{class_id}"
                print(f"  {j+1}. {class_name}: {conf:.3f}")
        else:
            print(f"\nImage {i+1}: No detections found")
        
        print("-" * 50)
else:
    print("❌ No trained model available for inference")

## Export YOLOv12 Model for Deployment

Export the trained model in different formats for production deployment.

In [ ]:
if 'best_model_path' in locals() and os.path.exists(best_model_path):
    print("📦 Exporting YOLOv12 Model for Deployment")
    
    model = YOLO(best_model_path)
    
    # Export to ONNX (recommended for deployment)
    try:
        print("Exporting to ONNX format...")
        onnx_path = model.export(
            format='onnx',
            imgsz=640,
            dynamic=False,  # Static shapes for better compatibility
            simplify=True,
            opset=12,       # Standard opset for compatibility
            half=False      # Keep FP32 for better compatibility
        )
        print(f"✅ ONNX export successful: {onnx_path}")
    except Exception as e:
        print(f"❌ ONNX export failed: {e}")
    
    # Export to TensorRT (for NVIDIA GPUs) - Optional
    if torch.cuda.is_available():
        try:
            print("Exporting to TensorRT format...")
            trt_path = model.export(
                format='engine',
                imgsz=640,
                half=True,
                dynamic=False,
                workspace=4  # 4GB workspace
            )
            print(f"✅ TensorRT export successful: {trt_path}")
        except Exception as e:
            print(f"⚠️ TensorRT export failed (this is normal if TensorRT is not installed): {e}")
    
    print("\n🚀 Model ready for deployment!")
else:
    print("❌ No trained model available for export")

## Model Performance Benchmark

Benchmark the trained YOLOv12 model to measure inference speed and performance.

In [ ]:
if 'best_model_path' in locals() and os.path.exists(best_model_path):
    print("⚡ YOLOv12 Performance Benchmark")
    
    model = YOLO(best_model_path)
    test_images_path = f"{dataset_path}/test/images"
    
    if os.path.exists(test_images_path):
        test_images = [f for f in os.listdir(test_images_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        # Select subset for benchmarking
        benchmark_images = random.sample(test_images, min(50, len(test_images)))
        
        print(f"Benchmarking on {len(benchmark_images)} images...")
        
        # Warm up
        warmup_img = f"{test_images_path}/{benchmark_images[0]}"
        for _ in range(5):
            _ = model(warmup_img, verbose=False)
        
        # Benchmark
        inference_times = []
        total_detections = 0
        
        start_time = time.time()
        
        for i, img_name in enumerate(benchmark_images):
            img_path = f"{test_images_path}/{img_name}"
            
            # Measure inference time
            inf_start = time.time()
            results = model(img_path, conf=0.25, iou=0.7, verbose=False)[0]
            inf_time = time.time() - inf_start
            
            inference_times.append(inf_time)
            total_detections += len(results.boxes) if results.boxes is not None else 0
            
            if (i + 1) % 10 == 0:
                print(f"  Processed {i + 1}/{len(benchmark_images)} images")
        
        total_time = time.time() - start_time
        
        # Calculate statistics
        avg_inference_time = np.mean(inference_times)
        std_inference_time = np.std(inference_times)
        avg_fps = 1.0 / avg_inference_time
        avg_detections = total_detections / len(benchmark_images)
        
        print(f"\n📊 BENCHMARK RESULTS:")
        print(f"Images processed: {len(benchmark_images)}")
        print(f"Total time: {total_time:.2f}s")
        print(f"Average inference time: {avg_inference_time*1000:.2f} ± {std_inference_time*1000:.2f} ms")
        print(f"Average FPS: {avg_fps:.2f}")
        print(f"Min inference time: {min(inference_times)*1000:.2f} ms")
        print(f"Max inference time: {max(inference_times)*1000:.2f} ms")
        print(f"Total detections: {total_detections}")
        print(f"Average detections per image: {avg_detections:.2f}")
        
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.max_memory_allocated() / 1024**3
            print(f"Peak GPU memory: {gpu_memory:.2f} GB")
    
    else:
        print("No test images found for benchmarking")
else:
    print("❌ No trained model available for benchmarking")

## Training Summary and Next Steps

Congratulations! You've successfully trained a YOLOv12 model for Bangladesh street food detection.

In [ ]:
print("🎉 YOLOv12 Bangladesh Street Food Detection - Training Complete!")
print("=" * 70)

if 'best_model_path' in locals() and os.path.exists(best_model_path):
    print(f"\n📋 PROJECT SUMMARY:")
    print(f"Model Architecture: YOLOv12 ({YOLOV12_CONFIG['model_size']})")
    print(f"Dataset Classes: {SELECTED_CLASSES}")
    print(f"Training Epochs: {YOLOV12_CONFIG['epochs']}")
    print(f"Batch Size: {YOLOV12_CONFIG['batch_size']}")
    print(f"Image Size: {YOLOV12_CONFIG['imgsz']}")
    
    print(f"\n💾 MODEL OUTPUTS:")
    print(f"Best Model: {best_model_path}")
    print(f"Results Directory: {YOLOV12_CONFIG['project']}/{YOLOV12_CONFIG['name']}")
    
    print(f"\n🚀 NEXT STEPS:")
    print(f"1. Test model on new Bangladesh street food images")
    print(f"2. Deploy using exported ONNX format")
    print(f"3. Create mobile or web application")
    print(f"4. Consider expanding to more street food classes")
    print(f"5. Fine-tune on additional regional variations")
    
    print(f"\n🎯 Your YOLOv12 model is ready to detect:")
    for i, food in enumerate(SELECTED_CLASSES, 1):
        print(f"   {i}. {food}")
        
else:
    print("\n❌ Training was not completed successfully.")
    print("Please check the error messages above and retry.")

print(f"\n{'='*70}")
print("🍛 Happy Bangladesh Street Food Detection with YOLOv12! 🍛")
print(f"{'='*70}")